In [4]:
"""
Rebuild the slim, cleaned dataset straight from the original CSV.
We need a slimmed dataset as we were getting kernel crashes from using too much memory.
Once loaded, the original CSV was 1.5GB, whereas the slimmed .pkl only takes 0.48GB
Output: hi_small_slim_usd.pkl
"""
import os, glob, time, gc
import numpy as np
import pandas as pd
from pandas.api.types import union_categoricals

SHARED = os.path.expanduser('~/shared')
OUT = f'{SHARED}/hi_small_slim_usd.pkl'

# ---------------------------------------------------------------- find the CSV
candidates = glob.glob(os.path.expanduser('~/**/HI-Small_Trans.csv'), recursive=True)
if not candidates:
    raise FileNotFoundError('HI-Small_Trans.csv not found under your home folder. '
                            'Upload it to ~/shared/ and run again.')
CSV = candidates[0]
print('reading', CSV)

# ---------------------------------------------------------------- load compactly
t0 = time.time()
names = ['timestamp', 'from_bank', 'from_account', 'to_bank', 'to_account',
         'amount_received', 'receiving_currency', 'amount_paid', 'payment_currency',
         'payment_format', 'is_laundering']
df = pd.read_csv(
    CSV, header=0, names=names,
    dtype={'from_bank': 'int64', 'to_bank': 'int64',
           'from_account': 'category', 'to_account': 'category',
           'amount_received': 'float64', 'amount_paid': 'float64',
           'receiving_currency': 'category', 'payment_currency': 'category',
           'payment_format': 'category', 'is_laundering': 'int8'},
)
df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y/%m/%d %H:%M')
print(df.shape, f'loaded in {time.time() - t0:.1f}s',
      f'| {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# ---------------------------------------------------------------- account IDs (no big text)
# An account is (bank, account number). Build one number per account without
# creating millions of text strings, then make readable labels only for the
# ~500k unique accounts.
acc = union_categoricals([df['from_account'], df['to_account']]).categories
from_code = pd.Categorical(df['from_account'], categories=acc).codes.astype('int64')
to_code = pd.Categorical(df['to_account'], categories=acc).codes.astype('int64')
n = len(acc)
key_s = df['from_bank'].values * n + from_code
key_r = df['to_bank'].values * n + to_code
ids, uniq = pd.factorize(np.concatenate([key_s, key_r]))
labels = [f'{b}_{a}' for b, a in zip(uniq // n, acc[uniq % n])]   # e.g. "3209_8000F4670"

df['sender'] = pd.Categorical.from_codes(ids[:len(df)], categories=labels)
df['receiver'] = pd.Categorical.from_codes(ids[len(df):], categories=labels)
df['s_id'] = ids[:len(df)].astype('int64')
df['r_id'] = ids[len(df):].astype('int64')
df = df.drop(columns=['from_account', 'to_account'])
del acc, from_code, to_code, key_s, key_r, ids, uniq, labels
gc.collect()
print(f'account IDs built | {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

# ---------------------------------------------------------------- clean
keep = (df['s_id'] != df['r_id']) & (df['timestamp'] < '2022-09-11')
df = df[keep].sort_values('timestamp', kind='stable').reset_index(drop=True)
del keep
df['t'] = (df['timestamp'] - df['timestamp'].min()).dt.total_seconds().astype('int64')

# ---------------------------------------------------------------- USD amounts
usd_rate = {
    'US Dollar':         1.0,
    'Bitcoin':           11881.623158,
    'UK Pound':          1.291656,
    'Euro':              1.171783,
    'Swiss Franc':       1.092896,
    'Canadian Dollar':   0.757978,
    'Australian Dollar': 0.707814,
    'Shekel':            0.296121,
    'Saudi Riyal':       0.266588,
    'Brazil Real':       0.177101,
    'Yuan':              0.149307,
    'Mexican Peso':      0.047297,
    'Rupee':             0.013616,
    'Ruble':             0.012853,
    'Yen':               0.009488,
}
df['amount_paid_usd'] = (df['amount_paid'] *
                         df['payment_currency'].map(usd_rate).astype('float32')).astype('float32')
df['amount_received_usd'] = (df['amount_received'] *
                             df['receiving_currency'].map(usd_rate).astype('float32')).astype('float32')
assert df[['amount_paid_usd', 'amount_received_usd']].notna().all().all(), 'a currency is missing from usd_rate'

# ---------------------------------------------------------------- check + save safely
print(df.shape, '| laundering:', int(df['is_laundering'].sum()),
      f"({df['is_laundering'].mean():.3%})",
      f"| {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")
print('expected roughly: (4486418, ...) | laundering: 4514 (0.101%)')

tmp = OUT + '.tmp'
df.to_pickle(tmp)
os.replace(tmp, OUT)          # rename is instant, so a crash can't leave a half-written file
print('saved', OUT, f'| total time {time.time() - t0:.1f}s')

reading /home/ibm-user/HI-Small_Trans.csv
(5078345, 11) loaded in 15.2s | 0.35 GB
account IDs built | 0.45 GB
(4486418, 16) | laundering: 4514 (0.101%) | 0.48 GB
expected roughly: (4486418, ...) | laundering: 4514 (0.101%)
saved /home/ibm-user/shared/hi_small_slim_usd.pkl | total time 20.8s
